# CodeBlooded v3: training that produced the shipped model (Kaggle T4, about 35 min incl. scoring)

**Settings:** Accelerator **GPU T4 x2**, Internet **On** (clone + pip only), Add Input: a dataset with the kit's
`india_climate_train.nc` and `stats.json` (v1.0.0). Then **Run All**.

Cells 1-4 get the code and check the data. Cells 5-10 train v3 (same architecture as the served model), pick the
checkpoint by the grader's score on 2019, write the bundle, score it on held-out 2020-21, fit and score the
climatology pull, and zip everything. To ship: copy `bundle_v3/*` into `server/bundle/` and rename
`clim_harmonics_v3.npy` -> `clim_harmonics.npy`, `clim_pull_v3.npy` -> `clim_pull.npy`.

In [ ]:
# 1. Configuration
REPO = "https://github.com/Bhoomified/CodeBlooded.git"
BRANCH = "main"

In [ ]:
# 2. Get the code (weights are not needed for training, so skip any Git LFS download)
import os
os.environ["GIT_LFS_SKIP_SMUDGE"] = "1"
%cd /kaggle/working
!rm -rf CodeBlooded && git clone --depth 1 -b $BRANCH $REPO
%cd /kaggle/working/CodeBlooded
!git log --oneline -1 | tee /kaggle/working/commit.txt

In [ ]:
# 3. Install what Kaggle lacks and check the GPU
!pip install -q netCDF4 "monsoon-r1-kit @ git+https://github.com/OfficialAbhinavSingh/yug-monsoon-kit@v1.0.0"
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: Settings -> Accelerator -> GPU T4 x2, then restart"
cap = torch.cuda.get_device_capability(0)
print(torch.cuda.get_device_name(0), cap)
assert cap >= (7, 0), "This is a P100 (or older): switch to GPU T4 x2"

In [ ]:
# 4. Find the input files and check them
import glob, json, xarray as xr, numpy as np
nc = glob.glob("/kaggle/input/**/india_climate_train.nc", recursive=True)
st = glob.glob("/kaggle/input/**/stats.json", recursive=True)
assert nc, "india_climate_train.nc not found: Add Input -> your dataset with the kit files"
assert st, "stats.json not found: add it to the same dataset"
DATA, STATS = nc[0], st[0]
ds = xr.open_dataset(DATA)
print(DATA, ds["data"].shape, str(ds.time.values[0])[:10], "->", str(ds.time.values[-1])[:10])
assert list(ds["channel"].values) == ["t2m", "tp", "u10", "v10", "msl", "tcwv"], "channel order"
assert ds["lat"].values[0] < ds["lat"].values[-1], "latitude must run south to north"
assert np.isfinite(ds["data"].values).all(), "NaN in data"
assert len(json.load(open(STATS))["std"]) == 6
print("input checks passed")

In [ ]:
# 5. v3 SETUP: data, normalisation, climatology and a 2019 validation scorer that uses the kit's own formulas.
#     Same architecture and inputs as v1/v2 (speed-limited residual CNN), so server/core.py can serve v3 unchanged.
import os, sys, json, math, time, copy, glob, shutil, datetime as dt
import numpy as np, torch, xarray as xr
WORK = os.environ.get("CB_WORK", "/kaggle/working")
REPO_DIR = os.environ.get("CB_REPO", f"{WORK}/CodeBlooded")
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
from train.pipeline import load, prep, window_starts, theta_of, train_ridge, TRAIN_YEARS, VAL_YEARS
from server.core import Norm, make_static, build_net, STATE_CH, TP, NC, NLAT, NLON
from monsoon_r1 import scoring as S, stats as KS
from monsoon_r1.contract import LEAD_DAYS

DATA = glob.glob(os.environ.get("CB_DATA", "/kaggle/input/**/india_climate_train.nc"), recursive=True)[0]
STATS = glob.glob(os.environ.get("CB_STATS", "/kaggle/input/**/stats.json"), recursive=True)[0]
DEV = "cuda" if torch.cuda.is_available() else "cpu"
QUICK = bool(int(os.environ.get("CB_QUICK", "0")))     # 1 = tiny CPU smoke run (only for testing the cells)

X, SRAW, YEARS, DOY = load(DATA)
_t = xr.open_dataset(DATA)["time"].values.astype("datetime64[D]")
DATES = np.array([int(str(d).replace("-", "")) for d in _t])
NORM = prep(X, YEARS)                       # fitted on 2005-2018 only
ZN = Norm(NORM).fwd(X)
STATIC5 = make_static(SRAW)
STD = np.array(json.load(open(STATS))["std"], "f4")
TR = (YEARS >= TRAIN_YEARS[0]) & (YEARS <= TRAIN_YEARS[1])
CLIM = KS.climatology(X[TR], DATES[TR])     # grader's climatology reference (train years)
def _date(i): d = int(DATES[i]); return dt.date(d // 10000, d // 100 % 100, d % 100)
def clim_at(i): return KS.clim_for(CLIM, _date(i))

VAL_STARTS = window_starts(YEARS, VAL_YEARS, 10)[::(12 if QUICK else 3)]   # 2019 only (2020-21 stays untouched)
VAL_CLIM = {L: np.stack([clim_at(t + L) for t in VAL_STARTS]) for L in LEAD_DAYS}
print(f"device {DEV} | train windows from {TRAIN_YEARS} | {len(VAL_STARTS)} validation windows in 2019")


def kit_points(pred, starts, clim_by_lead):
    """pred [n,10,6,32,36] physical units -> grader's forecast points (30 RMSE skill + 15 CSI skill) and anti-blur."""
    ss_w, csi_w, ratios = [], [], []
    for j, t in enumerate(starts):
        sv, cm, cc, cp, cl = [], [], [], [], []
        for L in LEAD_DAYS:
            true, pers, c = X[t + L], X[t], clim_by_lead[L][j]
            sv.append(np.nanmean(S.skill_vs_best_ref(pred[j, L - 1][None], true[None], pers[None], c[None])["ss"]))
            cm.append(pred[j, L - 1, TP]); cc.append(true[TP]); cp.append(pers[TP]); cl.append(c[TP])
        ss_w.append(float(np.mean(sv)))
        tm = np.stack(cc)
        csi_w.append(S.csi_skill(S.csi(np.stack(cm), tm), S.csi(np.stack(cp), tm), S.csi(np.stack(cl), tm)))
        ratios.append(S.variance_ratio(pred[j, 7], X[t + 8]))
    fp = S.forecast_points(ss_w, csi_w)
    blur = S.anti_blur_points(np.array(ratios))
    return {"fc45": fp["points"], "rmse30": fp["rmse_pts"], "csi15": fp["csi_pts"], "ss": fp["mean_ss"],
            "csi_skill": fp["mean_csi_skill"], "blur10": blur, "score55": fp["points"] + blur}

In [ ]:
# 6. v3 TRAINING. What is new vs v1/v2 (all training-side; the served model and inputs are unchanged):
#   a) checkpoint selection by the grader's score on 2019 (not by loss): we keep the weights that score best
#   b) loss weighted towards what is scored: lead days 1/4/8 count double, winds and rain get more weight
#   c) EMA (moving average) of the weights: smoother, generalises better, costs nothing at inference
#   d) small input noise during multi-day training: the model learns to correct its own errors (less drift)
#   e) mixed precision on the GPU: about 1.5-2x faster training on a T4
CFG = dict(
    width=96, blocks=4,                          # keep: wider/deeper = more overfitting, deeper = wider reach per day
    epochs1=30, lr1=3e-4,                        # 1-day phase
    plan=[(4, 10, 1e-4), (8, 12, 3e-5)],         # (days per rollout, epochs, learning rate)
    batch=32,
    ch_w=[1.0, 1.4, 1.4, 1.0, 1.0],              # state channels t2m, u10, v10, msl, tcwv
    rain_w=1.4, heavy_w=2.0,                     # rain head weight; extra weight on >10 mm/day cells
    scored_lead_w=2.0,                           # lead days 1, 4, 8 count this much more
    grad_w=0.1, noise=0.02, ema=0.999, seed=0,
    val_every1=5,                                # validate every N epochs in the 1-day phase (every epoch later)
)
if QUICK:
    CFG.update(width=16, epochs1=1, plan=[(4, 1, 1e-4)], val_every1=1)


def train_v3(cfg, limit=None):
    torch.manual_seed(cfg["seed"]); rng = np.random.default_rng(cfg["seed"])
    net = build_net(width=cfg["width"], blocks=cfg["blocks"]).to(DEV)
    ema = copy.deepcopy(net).eval()
    for p in ema.parameters(): p.requires_grad_(False)
    amp = DEV == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp) if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler") else torch.cuda.amp.GradScaler(enabled=amp)
    SC = list(STATE_CH)
    dsd = torch.tensor(NORM["dsd"], dtype=torch.float32, device=DEV)[SC][:, None, None]
    chw = torch.tensor(cfg["ch_w"], dtype=torch.float32, device=DEV)[:, None, None]
    latw = torch.cos(torch.deg2rad(torch.linspace(6.5, 37.5, NLAT, device=DEV))); latw = (latw / latw.mean())[:, None]
    Zt = torch.from_numpy(ZN).to(DEV); stat = torch.from_numpy(STATIC5).to(DEV)
    mu = torch.tensor(NORM["mu"], device=DEV)[:, None, None]; sd = torch.tensor(NORM["sd"], device=DEV)[:, None, None]
    heavy_z = (math.log1p(10.0) - NORM["mu"][TP]) / NORM["sd"][TP]
    rain_floor = (0.0 - NORM["mu"][TP]) / NORM["sd"][TP]

    def inputs(z3, th):
        n = z3.shape[0]
        s = torch.stack([torch.sin(th), torch.cos(th)], 1)[:, :, None, None].expand(n, 2, NLAT, NLON)
        return torch.cat([z3.reshape(n, 18, NLAT, NLON), stat[None].expand(n, -1, -1, -1), s], 1)

    def step(model, z3, th):
        ds, rn = model(inputs(z3, th))
        ds, rn = ds.float(), rn.float()
        nxt = z3[:, -1].clone()
        nxt[:, SC] = nxt[:, SC] + ds * dsd
        nxt[:, TP] = torch.clamp(rn[:, 0], min=rain_floor)
        return nxt, ds, rn[:, 0]

    def loss_fn(z_last, nxt, ds, rn, target, lw):
        tgt_ds = (target[:, SC] - z_last[:, SC]) / dsd
        l_state = (((ds - tgt_ds) ** 2) * chw * latw).mean()
        w = 1.0 + cfg["heavy_w"] * (target[:, TP] > heavy_z).float()
        l_rain = cfg["rain_w"] * (((rn - target[:, TP]) ** 2) * w * latw).mean()
        gx = lambda a: a[..., :, 1:] - a[..., :, :-1]; gy = lambda a: a[..., 1:, :] - a[..., :-1, :]
        l_grad = (gx(nxt) - gx(target)).abs().mean() + (gy(nxt) - gy(target)).abs().mean()
        return lw * (l_state + l_rain + cfg["grad_w"] * l_grad)

    @torch.no_grad()
    def rollout_phys(model, starts, steps=10, bs=64):
        """Same update as server/core.py (rain fed back after clipping to >= 0). Returns physical units."""
        outs = []
        for b in range(0, len(starts), bs):
            idx = starts[b:b + bs]
            z3 = torch.stack([Zt[i - 2:i + 1] for i in idx])
            th = torch.tensor([theta_of(DOY[i] + 1) for i in idx], dtype=torch.float32, device=DEV)
            seq = []
            for k in range(steps):
                nxt, _, _ = step(model, z3, th)
                x = nxt * sd + mu
                x[:, TP] = torch.expm1(torch.clamp(x[:, TP], 0, 8.0))
                seq.append(x.cpu().numpy())
                fb = x.clone(); fb[:, TP] = torch.log1p(fb[:, TP]); fb = (fb - mu) / sd
                z3 = torch.cat([z3[:, 1:], fb[:, None]], 1)
                th = th + 2 * math.pi / 365.0
            outs.append(np.stack(seq, 1))
        return np.concatenate(outs, 0)

    best = {"score55": -1e9}; hist = []

    def validate(tag):
        nonlocal best
        m = kit_points(rollout_phys(ema, VAL_STARTS), VAL_STARTS, VAL_CLIM)
        hist.append({"tag": tag, **m})
        flag = ""
        if m["score55"] > best["score55"]:
            best = {**m, "tag": tag, "state": {k: v.detach().cpu().clone() for k, v in ema.state_dict().items()}}
            flag = "  <- best so far"
        print(f"    val 2019 [{tag}] forecast {m['fc45']:.2f}/45 (rmse {m['rmse30']:.2f}, csi {m['csi15']:.2f}) "
              f"blur {m['blur10']:.2f}/10 | skill {m['ss']:+.3f} csiS {m['csi_skill']:+.3f}{flag}", flush=True)

    def run(k_steps, epochs, lr, val_every):
        starts = window_starts(YEARS, TRAIN_YEARS, k_steps)
        if limit: starts = rng.choice(starts, size=min(limit, len(starts)), replace=False)
        nb = max(1, len(starts) // cfg["batch"])
        opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-5)
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=max(1, epochs * nb), pct_start=0.1)
        lws = [cfg["scored_lead_w"] if (k + 1) in LEAD_DAYS else 1.0 for k in range(k_steps)]
        for ep in range(epochs):
            net.train(); perm = rng.permutation(starts); tot, t0 = 0.0, time.time()
            for b in range(nb):
                idx = perm[b * cfg["batch"]:(b + 1) * cfg["batch"]].tolist()
                z3 = torch.stack([Zt[i - 2:i + 1] for i in idx])
                th = torch.tensor([theta_of(DOY[i] + 1) for i in idx], dtype=torch.float32, device=DEV)
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
                    loss = 0.0
                    for k in range(k_steps):
                        target = torch.stack([Zt[i + 1 + k] for i in idx])
                        zin = z3 + cfg["noise"] * torch.randn_like(z3) if (k_steps > 1 and cfg["noise"] > 0) else z3
                        nxt, ds, rn = step(net, zin, th)
                        loss = loss + loss_fn(z3[:, -1], nxt, ds, rn, target, lws[k])
                        z3 = torch.cat([z3[:, 1:], nxt[:, None]], 1)
                        th = th + 2 * math.pi / 365.0
                    loss = loss / sum(lws)
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
                scaler.step(opt); scaler.update(); sched.step()
                with torch.no_grad():
                    for pe, p in zip(ema.parameters(), net.parameters()):
                        pe.mul_(cfg["ema"]).add_(p.detach(), alpha=1 - cfg["ema"])
                tot += loss.item()
            print(f"  rollout {k_steps} | epoch {ep + 1}/{epochs} | loss {tot / nb:.4f} | {time.time() - t0:.0f}s", flush=True)
            if (ep + 1) % val_every == 0 or ep + 1 == epochs:
                validate(f"{k_steps}d ep{ep + 1}")
        torch.save(best["state"], f"{WORK}/cnn_v3_best_partial.pt")   # survives a session cut

    t_all = time.time()
    run(1, cfg["epochs1"], cfg["lr1"], cfg["val_every1"])
    for k_steps, epochs, lr in cfg["plan"]:
        run(k_steps, epochs, lr, 1)
    print(f"done in {time.time() - t_all:.0f}s | best checkpoint: {best['tag']} | val 2019 forecast {best['fc45']:.2f}/45, "
          f"blur {best['blur10']:.2f}/10")
    return best, hist


BEST, HIST = train_v3(CFG, limit=256 if QUICK else None)

In [ ]:
# 7. Write the v3 bundle in the exact format server/core.py loads (no server change needed)
B3 = f"{WORK}/bundle_v3"
os.makedirs(B3, exist_ok=True)
torch.save(BEST["state"], f"{B3}/cnn.pt")
np.save(f"{B3}/static.npy", SRAW)
meta = {"name": f"speed-limited-cnn-w{CFG['width']}-b{CFG['blocks']}-v3", "norm": NORM,
        "net": {"width": CFG["width"], "blocks": CFG["blocks"]}, "quick": QUICK}
json.dump(meta, open(f"{B3}/meta.json", "w"))
old = f"{WORK}/bundle/ridge.npz"
if os.path.exists(old) and os.path.getsize(old) > 1024:
    shutil.copy(old, f"{B3}/ridge.npz")                                     # same data + normalisation = same ridge
else:
    np.savez(f"{B3}/ridge.npz", W=train_ridge(ZN, STATIC5, DOY, window_starts(YEARS, TRAIN_YEARS, 1)).astype("f8"))
qsrc = f"{WORK}/bundle/qm.json" if os.path.exists(f"{WORK}/bundle/qm.json") else "server/bundle/qm.json"
q = json.load(open(qsrc)); q["alpha"] = [[0.0] * len(r) for r in q["alpha"]]   # equaliser off (it never helped)
json.dump(q, open(f"{B3}/qm.json", "w"))
json.dump({"cfg": CFG, "best": {k: v for k, v in BEST.items() if k != "state"}, "history": HIST},
          open(f"{WORK}/train_v3_history.json", "w"), indent=1)
print(sorted(os.listdir(B3)), "| best:", BEST["tag"])

In [ ]:
# 8. Full grader replica on held-out 2020-21 (forecast + anti-blur + causality shocks), then compare with v2
EVERY = 60 if QUICK else 5
!python eval/local_score.py --data "$DATA" --stats "$STATS" --bundle "$B3" --every $EVERY --out "$WORK/results_v3.json" 2>&1 | tail -9
def _cnn_row(path):
    r = {x["model"]: x for x in json.load(open(path))}
    return r["CNN raw"]
rows = [("v3", f"{WORK}/results_v3.json")]
for name, p in (("v2", f"{WORK}/results.json"), ("v2 (repo)", "eval/results.json")):
    if os.path.exists(p): rows.insert(0, (name, p)); break
print(f"\n{'model':10s}{'TOTAL/70':>10s}{'forecast':>10s}{'rmse30':>8s}{'csi15':>8s}{'blur10':>8s}{'caus15':>8s}{'skill':>8s}"
      f"  stall-propagates")
for name, p in rows:
    c = _cnn_row(p)
    print(f"{name:10s}{c['total_auto_70']:10.2f}{c['forecast_points_45']:10.2f}{c['rmse_points_30']:8.2f}{c['csi_points_15']:8.2f}"
          f"{c['anti_blur_points_10']:8.2f}{c['causality_points_15']:8.2f}{c['mean_skill']:+8.3f}  "
          f"{c['shock_pass_rates']['monsoon_stall']['propagates']}")

In [ ]:
# 9. EXPERIMENT: lead-dependent pull towards climatology (cheap post-processing, no training).
#     At long leads the best RMSE forecast leans towards the seasonal average. Strength per lead and channel is
#     fitted on 2019 only, never decreases with lead, and climatology comes from day-of-year harmonics
#     (exactly what the server could compute from doy_sincos, no dates). Scored on 2020-21 with the full grader.
from server.core import Simulator
import eval.local_score as LS
LS.STD[:] = STD
H = 3                                                                        # harmonics: annual + 2 overtones
day = np.arange(366); ang = 2 * np.pi * day / 365.0
A = np.stack([np.ones(366)] + [f(k * ang) for k in range(1, H + 1) for f in (np.sin, np.cos)], 1)   # [366, 7]
COEF = np.linalg.lstsq(A, CLIM.reshape(366, -1), rcond=None)[0].reshape(-1, NC, NLAT, NLON).astype("f4")
def clim_theta(theta):
    b = np.array([1.0] + [f(k * theta) for k in range(1, H + 1) for f in (math.sin, math.cos)], "f4")
    return np.tensordot(b, COEF, 1)

sim3 = Simulator(B3)
def raw_fc(x3, d, steps, pert):
    return sim3.simulate(x3, LS.sincos(d), steps, pert)[0]

# fit strengths on 2019
GRID = np.array([0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6])
lw = S.lat_weights()[:, None]
P, O, Cl = [], [], []
for t in VAL_STARTS:
    th0 = 2 * math.pi * DOY[t] / 365.0
    P.append(raw_fc(X[t - 2:t + 1], DOY[t], 10, None)); O.append(X[t + 1:t + 11])
    Cl.append(np.stack([clim_theta(th0 + 2 * math.pi * (k + 1) / 365.0) for k in range(10)]))
P, O, Cl = map(np.stack, (P, O, Cl))
LAM = np.zeros((10, NC), "f4")
for k in range(10):
    for c in range(NC):
        errs = [np.sqrt((((P[:, k, c] + g * (Cl[:, k, c] - P[:, k, c]) - O[:, k, c]) ** 2) * lw).mean()) for g in GRID]
        LAM[k, c] = GRID[int(np.argmin(errs))]
LAM = np.maximum.accumulate(LAM, axis=0)                                     # never weaker at a longer lead
print("pull strength per lead (rows) and channel t2m tp u10 v10 msl tcwv:\n", LAM)

def damped_fc(x3, d, steps, pert):
    out = raw_fc(x3, d, steps, pert)
    th0 = 2 * math.pi * d / 365.0
    for k in range(steps):
        lam = LAM[min(k, 9)][:, None, None]
        out[k] = out[k] + lam * (clim_theta(th0 + 2 * math.pi * (k + 1) / 365.0) - out[k])
    out[:, TP] = np.clip(out[:, TP], 0, None)
    return out

Xs, dts, yrs, dys = LS.load(DATA)
starts = [t for t in range(2, len(yrs) - 10) if yrs[t - 2] >= 2020 and yrs[t + 10] <= 2021][::EVERY]
res_raw = LS.evaluate("v3 raw", raw_fc, Xs, dts, yrs, dys, CLIM, starts, starts[::6])
res_dmp = LS.evaluate("v3 + climatology pull", damped_fc, Xs, dts, yrs, dys, CLIM, starts, starts[::6])
for r in (res_raw, res_dmp):
    print(f"{r['model']:24s} total {r['total_auto_70']:6.2f}/70 | forecast {r['forecast_points_45']:5.2f} "
          f"(rmse {r['rmse_points_30']:5.2f}, csi {r['csi_points_15']:5.2f}) | blur {r['anti_blur_points_10']:5.2f} "
          f"| caus {r['causality_points_15']:5.2f} | null ok {r['null_identical_all']}")
np.save(f"{WORK}/clim_harmonics_v3.npy", COEF); np.save(f"{WORK}/clim_pull_v3.npy", LAM)
json.dump([res_raw, res_dmp], open(f"{WORK}/results_v3_climpull.json", "w"), indent=1)
print("KEEP the pull" if res_dmp["total_auto_70"] > res_raw["total_auto_70"] + 0.2 else "DROP the pull (not clearly better)")

In [ ]:
# 10. Package everything for review (do NOT commit to the repo yet)
!cd "$WORK" && rm -f codeblooded_v3_candidate.zip && zip -qr codeblooded_v3_candidate.zip bundle_v3 results_v3.json train_v3_history.json results_v3_climpull.json clim_harmonics_v3.npy clim_pull_v3.npy && ls -la codeblooded_v3_candidate.zip